# 01 - Dataset inventory
Gets the Olist data onto disk and checks that all nine CSV files are there.

Order of attempts: existing CSVs, then a local `olist.zip`, then a Kaggle download.

In [ ]:
import os
import sys
import shutil
import zipfile
import subprocess
import importlib.util
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)

In [ ]:
# works whether the notebook is run from the project root or from notebooks/
here = Path.cwd().resolve()
PROJECT_ROOT = here.parent if here.name.lower() == "notebooks" else here

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder: ", DATA_DIR)

In [ ]:
EXPECTED_FILES = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]

def missing_csvs():
    return [f for f in EXPECTED_FILES if not (DATA_DIR / f).exists()]

def local_zips():
    # a zip either inside the data folder or next to the project root
    return sorted(set(DATA_DIR.glob("*.zip")) | set(PROJECT_ROOT.glob("olist*.zip")))

def pip_install(package, import_name=None):
    # installs into the active venv, only if the package is not there yet
    if importlib.util.find_spec(import_name or package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
        importlib.invalidate_caches()

print("Missing CSV files:", missing_csvs() or "none")

## Download (only if needed)

In [ ]:
DATASET_SLUG = "olistbr/brazilian-ecommerce"

if not missing_csvs():
    print("All CSV files already there, nothing to download.")

elif local_zips():
    print("Found a local zip, skipping the download:", [z.name for z in local_zips()])

else:
    pip_install("python-dotenv", "dotenv")
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")  # optional: KAGGLE_USERNAME / KAGGLE_KEY live here

    has_credentials = (
        (Path.home() / ".kaggle" / "kaggle.json").exists()
        or (os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"))
    )
    if not has_credentials:
        raise RuntimeError(
            "No Kaggle credentials found. Either put kaggle.json in your .kaggle folder, "
            "add KAGGLE_USERNAME / KAGGLE_KEY to a .env file, or drop olist.zip in the project root."
        )

    pip_install("kaggle")
    # the kaggle command sits next to python.exe inside the venv's Scripts folder
    kaggle_cli = shutil.which("kaggle") or str(Path(sys.executable).parent / "kaggle.exe")
    subprocess.run([kaggle_cli, "datasets", "download", DATASET_SLUG, "--path", str(DATA_DIR)], check=True)

## Extract

In [ ]:
# pulls the csv files out of any zip and drops them straight into DATA_DIR,
# even if the zip keeps them inside a sub-folder
if not missing_csvs():
    print("Nothing to extract.")
else:
    zips = local_zips()
    if not zips:
        raise FileNotFoundError(f"No zip found in {DATA_DIR} or {PROJECT_ROOT}.")

    for zip_path in zips:
        print("Extracting", zip_path.name)
        with zipfile.ZipFile(zip_path) as archive:
            for member in archive.namelist():
                if member.lower().endswith(".csv"):
                    with archive.open(member) as src, open(DATA_DIR / Path(member).name, "wb") as dst:
                        shutil.copyfileobj(src, dst)

## Verify and inventory

In [ ]:
import pandas as pd
from IPython.display import display

missing = missing_csvs()
if missing:
    raise RuntimeError(f"Still missing: {missing}")
print("All nine Olist CSV files are present.")

rows = []
for filename in EXPECTED_FILES:
    path = DATA_DIR / filename
    columns = pd.read_csv(path, nrows=1).columns
    with path.open("r", encoding="utf-8", errors="replace") as f:
        n_rows = sum(1 for _ in f) - 1  # minus the header line
    rows.append({
        "file": filename,
        "size_mb": round(path.stat().st_size / 1024**2, 2),
        "rows": n_rows,
        "columns": len(columns),
    })

inventory = pd.DataFrame(rows)
display(inventory)